In [ ]:
import numpy as np
ucs = -126/2
ui = -757/2
upb = -3482/4
uzn = -923

u = np.array([ucs, upb, ui])
n = np.array([8, 8, 24])

e_bulk = -16564

(e_bulk - np.sum(u*n))/np.sum(n)*13.6


std_form_energy = (e_bulk - np.sum(u*n))/np.sum(n)*13.6

In [66]:
if 'imports':
    from pymatgen.io.espresso.outputs import PWxml

from pathlib import Path
def get_element_formation_energy(elem, pseudo = 'pbe'):
    root = Path('/home/fes33/Documents/GIK - R&D/Personal - Papers and Reports/--Libraries/abinit/jhr/data/formation_energies')
    elem_file = root / elem / f"{elem}_{pseudo}.xml"

    comp_entry = PWxml(elem_file).get_computed_entry(entry_id = "")
    n_atoms = comp_entry.composition.as_data_dict()["unit_cell_composition"][elem]
    
    energy = comp_entry.energy
    
    en_per_atom = energy/n_atoms
    return en_per_atom
    

def fileread(func):
    def wrapper(cls, xml_file, *args, **kwargs):
        calc = PWxml(xml_file)
        struct = calc.get_computed_entry(entry_id = "")

        return func(cls, struct, *args, **kwargs)
    return wrapper
    


import numpy as np
class DefectTools:
    
    @classmethod
    @fileread
    def standardize_total_energy(cls, struct):
        e_bulk = struct.energy
        composition = struct.composition
        
        comp_dict = composition.as_data_dict()['unit_cell_composition']
        
        elements = comp_dict.keys()
        n_i = np.array(list(comp_dict.values()))
        u_i = np.array([get_element_formation_energy(elem) for elem in elements])
        
        print(n_i, elements, e_bulk)
        std_form_energy = (e_bulk - np.sum(n_i*u_i))/np.sum(n_i)

        return std_form_energy
    
    @classmethod
    @fileread
    def standardized_computed_entry(cls, struct):
        """
        
        """
        d_ = {
           "energy": cls.standardize_total_energy(struct),
           "composition": struct.composition,
           "entry_id": "",
           "correction": 0, #pristine_calc.get_computed_entry(entry_id = "").correction
        }
        print(d_)
        
        ent = ComputedEntry.from_dict(d_) #Computed entries list. Why twice?

xml_file = '/home/fes33/Documents/Temp/Mg_Ga/bulk_sc/aiida.xml'
#DefectTools.standardize_total_energy(xml_file) 
    
DefectTools.standardized_computed_entry(xml_file) 


Reading file: /home/fes33/Documents/Temp/Mg_Ga/bulk_sc/aiida.xml
Reading file:  ComputedStructureEntry - Cs8 Pb8 I24  (CsPbI3)
Energy (Uncorrected)     = -225371.7522 eV (-5634.2938 eV/atom)
Correction               = 0.0000    eV (0.0000   eV/atom)
Energy (Final)           = -225371.7522 eV (-5634.2938 eV/atom)
Energy Adjustments:
  None
Parameters:
  potcar_symbols         = ['Cs_pbe_v1.uspp.F.UPF Cs', 'I.pbe-n-kjpaw_psl.0.2.UPF I', 'Pb.pbe-dn-kjpaw_psl.0.2.2.UPF Pb']
  hubbards               = {}
  is_hubbard             = False
  run_type               = PBE
  potcar_spec            = ['Cs_pbe_v1.uspp.F.UPF', 'I.pbe-n-kjpaw_psl.0.2.UPF', 'Pb.pbe-dn-kjpaw_psl.0.2.2.UPF']
Data:


TypeError: expected str, bytes or os.PathLike object, not ComputedStructureEntry

In [ ]:
if 'imports':
    apikey = '4a3cEY6Q0G6DyR1BcQQ8UaDGK4JO3KFU'
    import sys
    import json
    #sys.path.append("/home/fes33/Documents/GIK - R&D/Personal - Papers and Reports/--Libraries/abinit/jhr/src/pymatgen-analysis-defects/")
    sys.path.append("/home/fes33/Documents/GIK - R&D/Personal - Papers and Reports/--Cloned/pymatgen-analysis-defects")
    # ------------------------
    from pymatgen.core import Element
    from mp_api.client import MPRester
    from pymatgen.io.espresso.outputs import PWxml
    
    
    import warnings
    from monty.serialization import loadfn, dumpfn
    # from pymatgen.analysis.defects.thermo import FormationEnergyDiagram
    from jhr.dev.defect.thermo import FormationEnergyDiagram
    from pymatgen.io.vasp import Vasprun
    
    warnings.filterwarnings("ignore")
    
    from pathlib import Path
    
    from pymatgen.analysis.defects.core import Substitution
    from pymatgen.core import PeriodicSite, Species, Structure
    
    #directory = "/home/fes33/Documents/GIK - R&D/Personal - Papers and Reports/--Libraries/abinit/jhr/src/pymatgen-analysis-defects/tests/test_files"
    directory = "/home/fes33/Documents/Temp/"
    
    TEST_FILES = Path(directory)